# mRNADesign
Single Colab entrypoint (NS-1.1). Run all cells top to bottom.
The cells below install dependencies and the package, then collect parameters.

In [ ]:
# 1. Install runtime dependencies (NS-3.* pinned versions).
!pip install -q pyyaml==6.0.3 pydantic==2.14.3 biopython==1.88


In [ ]:
# 2. Obtain the mrna_design package (VaxLab-style self-install).
# Set REPO_URL to your published GitHub repo once T69 (git) is done; otherwise
# upload mRNADesign.zip to /content and it will be unzipped automatically.
import os, sys, subprocess
REPO_URL = ""  # e.g. "https://github.com/<org>/mRNADesign.git"
REPO_DIR = "/content/mRNADesign"
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
if not os.path.isdir(REPO_DIR) and os.path.exists("/content/mRNADesign.zip"):
    subprocess.run(["unzip", "-q", "/content/mRNADesign.zip", "-d", "/content"], check=True)
if os.path.isdir(REPO_DIR) and REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if REPO_DIR not in sys.path and "/content" not in sys.path:
    sys.path.insert(0, "/content")
import mrna_design
print("mrna_design", mrna_design.__version__)


## Parameters
Fill in the form fields below, then **Runtime → Run all**.

In [ ]:
#@title Input
PROTEIN_SEQUENCE = "MKV" #@param {type:"string"}
RUN_MODULES = "design,construct" #@param {type:"string"}


In [ ]:
#@title Optimisation
TEMPLATE_TYPE = "pcr" #@param ["pcr", "plasmid"]
SEED = 42 #@param {type:"integer"}


In [ ]:
#@title QC
SKIP_GPU_TOOLS = False #@param {type:"boolean"}


In [ ]:
#@title Construct
BACKBONE_PATH = "" #@param {type:"string"}


In [ ]:
#@title Output
OUT_DIR = "/tmp/nb_run" #@param {type:"string"}


In [ ]:
from mrna_design.config import parse_config, resolve_config
from mrna_design.run import run_pipeline

run_modules = [m.strip() for m in RUN_MODULES.split(",") if m.strip()]
cfg = parse_config({
    "protein_sequence": PROTEIN_SEQUENCE,
    "run_modules": run_modules,
    "template_type": TEMPLATE_TYPE,
    "seed": SEED,
    "backbone_path": BACKBONE_PATH or None,
})
resolved = resolve_config(cfg)
result = run_pipeline(resolved.config if hasattr(resolved, "config") else cfg, OUT_DIR)
result.run_dir
